# Fine-tune DistilRoBERTa on the mental-health dataset

**Before running:** Runtime > Change runtime type > **T4 GPU**.

You will upload 3 files when asked:
1. `dataset.csv` (the Kaggle CSV)
2. `preprocessing.py` (the UPDATED version from `src/`)
3. `evaluate.py` (from `src/`)

The train/val/test split is identical to `src/train.py`, so the scores are directly comparable to your TF-IDF models. At the end you download `transformer_bundle.zip`.

In [ ]:
!nvidia-smi -L
!pip -q install -U transformers accelerate

In [ ]:
from google.colab import files
import shutil

print("Upload: dataset.csv, preprocessing.py, evaluate.py")
uploaded = files.upload()

# HuggingFace has its own package called `evaluate`, so copy ours under another name
shutil.copy("evaluate.py", "eval_utils.py")

In [ ]:
import json, os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from preprocessing import preprocess_series
from eval_utils import evaluate_predictions

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"

# ---- settings ----
MODEL_NAME = "distilroberta-base"      # for more accuracy (slower): "roberta-base"
DISPLAY_NAME = "DistilRoBERTa (fine-tuned)"
MAX_LEN = 256
EPOCHS = 3
LEARNING_RATE = 3e-5
BATCH_SIZE = 32
TEXT_COLUMN, TARGET_COLUMN = "statement", "status"
RANDOM_STATE, TEST_SIZE, VALIDATION_SIZE = 42, 0.2, 0.1

# ---- same cleaning as src/train.py ----
df = pd.read_csv("dataset.csv")
df = df.dropna(subset=[TEXT_COLUMN, TARGET_COLUMN]).copy()
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()
df = df[df[TEXT_COLUMN].str.len() > 0]
df = df.drop_duplicates(subset=[TEXT_COLUMN])
df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(str).str.strip().str.lower()
df = df.reset_index(drop=True)
df["clean_text"] = preprocess_series(df[TEXT_COLUMN])
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

le = LabelEncoder()
y = le.fit_transform(df[TARGET_COLUMN])
classes = list(le.classes_)
id2label = {i: c for i, c in enumerate(classes)}
label2id = {c: i for i, c in enumerate(classes)}

# ---- same split as src/train.py (the split depends only on y and the seed) ----
X_temp, X_test, y_temp, y_test = train_test_split(
    df[TEXT_COLUMN], y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
val_ratio = VALIDATION_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_ratio, random_state=RANDOM_STATE, stratify=y_temp)

print(classes)
print("Train / Val / Test:", len(X_train), len(X_val), len(X_test))
print("Expected from your log: 35706 / 5101 / 10202")

In [ ]:
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(texts):
    return tokenizer(list(texts), truncation=True, max_length=MAX_LEN)

class TextDS(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc, self.labels = enc, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: self.enc[k][i] for k in ("input_ids", "attention_mask")}
        item["labels"] = int(self.labels[i])
        return item

train_ds = TextDS(encode(X_train), y_train)
val_ds = TextDS(encode(X_val), y_val)
test_ds = TextDS(encode(X_test), y_test)

# Softened inverse-frequency class weights (helps the small classes / macro F1)
counts = np.bincount(y_train, minlength=len(classes))
weights = (len(y_train) / (len(classes) * counts)) ** 0.5
class_weights = torch.tensor(weights, dtype=torch.float32)
print("Class weights:", dict(zip(classes, weights.round(2))))

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(classes), id2label=id2label, label2id=label2id)

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "f1_macro": f1_score(p.label_ids, preds, average="macro", zero_division=0),
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits.float(), labels,
            weight=class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.06,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    fp16=True,
    report_to="none",
    seed=RANDOM_STATE,
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
# Best epoch (chosen on validation) is already loaded. Evaluate ONCE on the test set.
val_f1 = trainer.evaluate(val_ds)["eval_f1_macro"]

pred_out = trainer.predict(test_ds)
logits = torch.tensor(pred_out.predictions).float()
probs = torch.softmax(logits, dim=-1).numpy()
y_pred = probs.argmax(axis=1)

metrics = evaluate_predictions(y_test, y_pred, probs, classes=list(range(len(classes))))
metrics["classes"] = classes
metrics["per_class"] = {classes[int(k)]: v for k, v in metrics["per_class"].items()}
metrics["supports_probability"] = True
metrics["best_params"] = {"base_model": MODEL_NAME, "epochs": EPOCHS,
                          "max_len": MAX_LEN, "learning_rate": LEARNING_RATE}
metrics["val_f1_macro"] = float(val_f1)

print(f"TEST accuracy={metrics['accuracy']:.3f}  F1-macro={metrics['f1_macro']:.3f}  "
      f"F1-weighted={metrics['f1_weighted']:.3f}  (val F1-macro={val_f1:.3f})")
print("TF-IDF Logistic Regression for comparison: accuracy=0.769  F1-macro=0.720")

In [ ]:
# Export: transformer/ folder + transformer_metrics.json, zipped for download
shutil.rmtree("export", ignore_errors=True)
os.makedirs("export/transformer")
trainer.save_model("export/transformer")
tokenizer.save_pretrained("export/transformer")
with open("export/transformer_metrics.json", "w") as f:
    json.dump({DISPLAY_NAME: metrics}, f, indent=2)

shutil.make_archive("transformer_bundle", "zip", "export")
files.download("transformer_bundle.zip")